# trainer-subclass-extend — faded example 3: Subclass adds accuracy by calling super().validate() first

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `trainer-subclass-extend`. The last cell reports your progress on the `Trainer: subclass extend pattern` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: subclass extend pattern` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`trainer-subclass-extend`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "trainer-subclass-extend"
DD_SUBTOPIC = "Trainer: subclass extend pattern"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When overriding `validate()` in a Trainer subclass, calling `super().validate()` first ensures the base class appends the val loss to `self.history['val_loss']` before your extension runs. You can then safely read `self.history['val_loss'][-1]` in your extension without risk of an index error.

## Faded exercise 3

A `BaseTrainer` is provided. Implement `AccTrainer(BaseTrainer)` with `__init__` that calls `super().__init__(...)` and adds `self.acc_history = []`. Override `validate(self)`: first call `super().validate()` to populate the loss history, then do a second inference pass over `self.val_loader` to compute accuracy (argmax predictions vs integer labels), and append the float accuracy to `self.acc_history`. The blank is the `super().validate()` call.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import torch.nn as nn

class BaseTrainer:
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.step = 0
        self.history = {'train_loss': [], 'val_loss': []}

    def _step(self, x, y):
        return self.loss_fn(self.model(x), y)

    def fit(self, n_epochs):
        for _ in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()

    def validate(self):
        self.model.eval()
        total, count = 0.0, 0
        with t.inference_mode():
            for x, y in self.val_loader:
                loss = self.loss_fn(self.model(x), y)
                total += loss.item() * x.shape[0]
                count += x.shape[0]
        self.history['val_loss'].append(total / count)

class AccTrainer(BaseTrainer):
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        super().__init__(model, optimizer, train_loader, val_loader, loss_fn)
        self.acc_history = []

    def validate(self):
        raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above
        self.model.eval()
        correct, total = 0, 0
        with t.inference_mode():
            for x, y in self.val_loader:
                preds = self.model(x).argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.shape[0]
        self.acc_history.append(correct / total)


def _test():
    import torch as t
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    t.manual_seed(5)
    X = t.randn(30, 4)
    Y = (X[:, 0] > 0).long()
    train_dl = DataLoader(TensorDataset(X[:24], Y[:24]), batch_size=6)
    val_dl = DataLoader(TensorDataset(X[24:], Y[24:]), batch_size=6)
    model = nn.Linear(4, 2)
    opt = t.optim.SGD(model.parameters(), lr=0.05)
    trainer = AccTrainer(model, opt, train_dl, val_dl, nn.CrossEntropyLoss())
    trainer.fit(2)
    # Both histories should have 2 entries (one per epoch)
    assert len(trainer.history['val_loss']) == 2
    assert len(trainer.acc_history) == 2
    # Accuracy should be a float in [0, 1]
    for acc in trainer.acc_history:
        assert 0.0 <= acc <= 1.0, f'invalid accuracy: {acc}'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

class BaseTrainer:
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.step = 0
        self.history = {'train_loss': [], 'val_loss': []}

    def _step(self, x, y):
        return self.loss_fn(self.model(x), y)

    def fit(self, n_epochs):
        for _ in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()

    def validate(self):
        self.model.eval()
        total, count = 0.0, 0
        with t.inference_mode():
            for x, y in self.val_loader:
                loss = self.loss_fn(self.model(x), y)
                total += loss.item() * x.shape[0]
                count += x.shape[0]
        self.history['val_loss'].append(total / count)

class AccTrainer(BaseTrainer):
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        super().__init__(model, optimizer, train_loader, val_loader, loss_fn)
        self.acc_history = []

    def validate(self):
        super().validate()
        self.model.eval()
        correct, total = 0, 0
        with t.inference_mode():
            for x, y in self.val_loader:
                preds = self.model(x).argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.shape[0]
        self.acc_history.append(correct / total)
```
</details>